
# Silver Pedido Detalle

**Fuente:** `prod_dataco.brz_dataco.raw_dataco_supply_chain` (histórico completo acumulado)

**Destino:** Tabla de **Detalle de Pedido**, en `prod_dataco.slv_dataco`

**Grano:** un registro por `order_item_id` (línea de detalle de pedido)

## Qué hace

1. Lee todo el histórico de Bronze acumulado, no solo el último corte.
2. Deduplica por `order_item_id`, priorizando el registro correspondiente al corte más reciente mediante `dedup_reciente`, ordenando por `fecha_fin_corte` descendente.
3. Filtra `order_item_id` nulo, vacío o `'0'`, equivalente a la regla del SP original para `StagePedidoDetalle`.
4. Limpia y normaliza los datos:

   * Identificadores: `int`.
   * Cantidades: `int`.
   * Importes monetarios: `decimal(12,2)`.
   * `order_item_discount_rate`: `decimal(5,4)`.
   * `order_item_profit_ratio`: `decimal(9,6)`.
5. Conserva `order_id` para relacionar el detalle con la tabla de **Pedidos**.
6. Conserva las columnas de trazabilidad provenientes de Bronze.
7. Escribe la tabla Silver mediante `overwrite` completo.

## Columnas de negocio

* `order_item_id`
* `order_id`
* `order_item_cardprod_id`
* `order_item_quantity`
* `order_item_product_price`
* `order_item_discount`
* `order_item_discount_rate`
* `order_item_profit_ratio`
* `sales`
* `order_item_total`
* `order_profit_per_order`
* `benefit_per_order`
* `sales_per_customer`

## Trazabilidad conservada

* `fecha_inicio_corte`
* `fecha_fin_corte`
* `fecha_carga`
* `id_ejecucion`

## Nota de diseño

`order_id` se conserva en el detalle para permitir la relación con la tabla de **Pedidos**.

Esta relación es necesaria para los procesos posteriores de Gold que requieren combinar información del pedido con información a nivel de línea de detalle.

La tabla utiliza `dedup_reciente` para priorizar el registro correspondiente al corte más reciente cuando existen múltiples registros para un mismo `order_item_id`.

## Por qué importa este notebook

La tabla de **Detalle de Pedido** mantiene el nivel de granularidad de línea de pedido, sin realizar agregaciones.

Junto con la tabla de **Pedidos**, constituye la base para los indicadores de Gold que requieren información detallada de ventas, costos, descuentos y beneficios.

In [0]:
# ============================================================
# Celda 1: Parámetros + lectura de Bronze + helpers
# ------------------------------------------------------------
# Mismo patrón que silver_pedidos. Sesión propia, se repite todo.
# ============================================================

dbutils.widgets.text("bronze_catalog", "prod_dataco", "Catálogo Bronze")
dbutils.widgets.text("bronze_schema", "brz_dataco", "Esquema Bronze")
dbutils.widgets.text("bronze_table", "raw_dataco_supply_chain", "Tabla Bronze")

dbutils.widgets.text("silver_catalog", "prod_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "slv_dataco", "Esquema Silver")

bronze_catalog = dbutils.widgets.get("bronze_catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
bronze_table = dbutils.widgets.get("bronze_table")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")

BRONZE_TABLE_FQN = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"
SILVER_PEDIDO_DETALLE_FQN = f"{silver_catalog}.{silver_schema}.md_pedido_detalle"

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{silver_catalog}.{silver_schema}"
)

df_bronze = spark.table(BRONZE_TABLE_FQN)

from pyspark.sql import functions as F, Window

def clean_str(col_name: str, default: str = "") -> F.Column:
    return F.upper(F.trim(F.coalesce(F.col(col_name).cast("string"), F.lit(default))))

def clean_num(col_name: str, cast_type, default: str = "0") -> F.Column:
    trimmed = F.trim(F.col(col_name).cast("string"))
    nulled = F.when((trimmed.isNull()) | (trimmed == ""), None).otherwise(trimmed)
    return F.coalesce(nulled, F.lit(default)).cast(cast_type)

def dedup_reciente(df, partition_cols: list, orden_col: str):
    w = Window.partitionBy(*partition_cols).orderBy(F.col(orden_col).desc())
    return (
        df.withColumn("rn", F.row_number().over(w))
          .filter(F.col("rn") == 1)
          .drop("rn")
    )

print(f"Total filas en Bronze: {df_bronze.count()}")
print("✓ Helpers definidos: clean_str, clean_num, dedup_reciente")

In [0]:
[c for c in df_bronze.columns if "order_item" in c.lower() or c.lower() in ("sales", "order_profit_per_order", "benefit_per_order", "sales_per_customer")]

In [0]:
# ============================================================
# Celda 2: Deduplicación + construcción de hd_pedido_detalle
# ------------------------------------------------------------
# dedup por order_item_id, determinístico (fecha_fin_corte desc).
# Filtro: excluye order_item_id nulo/vacío/'0' -> misma regla que
# el SP original para StagePedidoDetalle.
#
# Se conservan las columnas de trazabilidad provenientes de Bronze
# para mantener la trazabilidad de extremo a extremo.
# ============================================================

df_detalle_dedup = dedup_reciente(
    df_bronze,
    ["order_item_id"],
    orden_col="fecha_fin_corte"
)

df_detalle_dedup = df_detalle_dedup.filter(
    F.col("order_item_id").isNotNull() &
    (F.trim(F.col("order_item_id").cast("string")) != "") &
    (F.trim(F.col("order_item_id").cast("string")) != "0")
)

df_silver_pedido_detalle = df_detalle_dedup.select(
    clean_num("order_item_id", "int").alias("order_item_id"),
    clean_num("order_id", "int").alias("order_id"),
    clean_num("order_item_cardprod_id", "int").alias("order_item_cardprod_id"),
    clean_num("order_item_quantity", "int").alias("order_item_quantity"),
    clean_num("order_item_product_price", "decimal(12,2)").alias("order_item_product_price"),
    clean_num("order_item_discount", "decimal(12,2)").alias("order_item_discount"),
    clean_num("order_item_discount_rate", "decimal(5,4)").alias("order_item_discount_rate"),
    clean_num("order_item_profit_ratio", "decimal(9,6)").alias("order_item_profit_ratio"),
    clean_num("sales", "decimal(12,2)").alias("sales"),
    clean_num("order_item_total", "decimal(12,2)").alias("order_item_total"),
    clean_num("order_profit_per_order", "decimal(12,2)").alias("order_profit_per_order"),
    clean_num("benefit_per_order", "decimal(12,2)").alias("benefit_per_order"),
    clean_num("sales_per_customer", "decimal(12,2)").alias("sales_per_customer"),

    # --------------------------------------------------------
    # Trazabilidad
    # --------------------------------------------------------
    F.col("fecha_inicio_corte"),
    F.col("fecha_fin_corte"),
    F.col("fecha_carga"),
    F.col("id_ejecucion")
)

print(f"Filas antes del dedup/filtro : {df_bronze.count()}")
print(f"Filas después                : {df_silver_pedido_detalle.count()}")

display(df_silver_pedido_detalle.limit(10))

In [0]:
# ============================================================
# Celda 3: Escritura a silver_pedido_detalle
# ------------------------------------------------------------
# overwrite completo, mismo patrón que el resto de Silver.
# ============================================================

(
    df_silver_pedido_detalle.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_PEDIDO_DETALLE_FQN)
)

print(f"✓ {df_silver_pedido_detalle.count()} filas escritas en {SILVER_PEDIDO_DETALLE_FQN}")